# Kilonova detection rates and area-distance figures

Per-run rate tables and area-distance figures of the three runs of the paper,
from the final tables in `results/`. The numbers, Table 5 and Figure 4 of the
paper come from `ugf-paper-results`, which the last section calls.


In [ ]:
import logging
import warnings
from pathlib import Path

import numpy as np
from astropy.table import QTable
from matplotlib import pyplot as plt

from ultrasat_gw_followup.analysis.area_distance import plot_area_distance
from ultrasat_gw_followup.analysis.selection_detection_rate import (
    summarize_selected_detected_events,
)
from ultrasat_gw_followup.probability.rate import (
    DEFAULT_POPULATION,
    MERGER_RATE,
    poisson_lognormal_rate_quantiles,
)

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
warnings.filterwarnings("ignore", ".*dubious year.*")
warnings.filterwarnings(
    "ignore", "Tried to get polar motions for times after IERS data is valid.*"
)

### Setup logging 

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    force=True,
)

In [ ]:
# Single source of truth for every path of this notebook. Each configuration
# names the final table of one run and where its table and figures are written.
ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
RESULTS = ROOT / "results"
OUTPUTS = ROOT / "paper_outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

# Campaign length of each O5 sub-period.
run_duration = 1  # years

# The three O5 sub-periods, in order.
O5_RUNS = ("O5a", "O5b", "O5c")

CONFIGS = {
    "ultrasat_lcs": {
        "events_file": RESULTS / "ultrasat_lcs.ecsv",
        "table_file": OUTPUTS / "ultrasat_lcs_detection.tex",
        "figure_prefix": OUTPUTS / "ultrasat_lcs_obj-vs-det",
    },
    "ultrasat_allsky": {
        "events_file": RESULTS / "ultrasat_allsky.ecsv",
        "table_file": OUTPUTS / "ultrasat_allsky_detection.tex",
        "figure_prefix": OUTPUTS / "ultrasat_allsky_obj-vs-det",
    },
    "uvex": {
        "events_file": RESULTS / "uvex.ecsv",
        "table_file": OUTPUTS / "uvex_detection.tex",
        "figure_prefix": OUTPUTS / "uvex_obj-vs-det",
    },
}


def prior_for(events_file):
    """Merger-rate prior of a table, from the model its metadata records."""
    model = QTable.read(events_file).meta.get("population_model", DEFAULT_POPULATION)
    return MERGER_RATE[model]


print(f"exports -> {OUTPUTS}")
for name, cfg in CONFIGS.items():
    status = "found" if cfg["events_file"].exists() else "MISSING"
    print(f"  {name:16s} {status:9s} {cfg['events_file'].relative_to(ROOT)}")

### Load simulated event data

In [ ]:
config = CONFIGS["ultrasat_lcs"]
main_table = QTable.read(config["events_file"])

config["events_file"]

### Get unique run names

In [ ]:
runs = np.unique(main_table["run"])
print(runs)

### Filter events by objective_value cutoff

In [ ]:
cutoff = main_table["cutoff"][0]
main_table = main_table[main_table["objective_value"] >= cutoff]
main_table[0:1]

### Group events by run

In [ ]:
event_tables_by_run = {run: main_table[main_table["run"] == run] for run in runs}
event_tables_by_run[runs[0]][0:1]

# Detection rates by run and source class

Expected number of selected and detected events per observing run (O5a, O5b, O5c) and source class (BNS, NSBH, All), from a Poisson-lognormal model.

- Selected: events whose scheduler objective reaches the cutoff.
- Detected: selected events weighted by the known-position detection probability.
- Credible intervals: 90%, from the lognormal merger-rate prior.

---

## 1 — Merger rate prior (GWTC-5.0)

Following Petrov et al. (`PeSi2021`) and Kiendrébéogo et al. (`Kiendrebeogo_2023`), we adopt the fiducial total CBC merger rate density from GWTC-5.0 (`gwtc5_population`):

$$\mathcal{R}_{\mathrm{CBC}}^{\mathrm{fid}} = 100^{+103}_{-50}\ \mathrm{Gpc^{-3}\,yr^{-1}} \qquad (90\%\ \mathrm{CI})$$

These are the FullPop numbers. The priors of both GWTC-5.0 models are declared once, in `ultrasat_gw_followup.probability.rate.MERGER_RATE`, and `prior_for()` above selects the right one from the population model the unpacker recorded in the table — so a sample and its rate prior cannot drift apart. PixelPop is 68 (+71, −29).

The declared quantiles are:

| Quantile | Rate |
|---|---|
| 5% | 50 $\mathrm{Gpc^{-3}\,yr^{-1}}$ |
| 50% (median) | 100 $\mathrm{Gpc^{-3}\,yr^{-1}}$ |
| 95% | 203 $\mathrm{Gpc^{-3}\,yr^{-1}}$ |

The lognormal prior is built from the median and from the 5–95% width, $\sigma = \log(203/50)/3.2897 = 0.426$. These three numbers are mutually consistent with a lognormal to better than 1%.

## 2 — Log-expected rate per run and source class

For each run $r$ and source class $c$ (BNS, NSBH, All), the log-expected number of events is:

$$\mu_{r,c} = \log(\text{target median}) + \log(\text{run duration}) - \log(\text{effective rate}_{\text{sim},r}) + \log(N_{r,c})$$

where $N_{r,c}$ is either the number of selected events or the sum of detection probabilities for class $c$ in run $r$. Note that the effective simulation rate depends only on the run, not the class.

## 3 — Poisson-lognormal quantiles

The 5%, 50%, and 95% quantiles of the rate are computed from the Poisson-lognormal model and reported as $\mathrm{median}_{-\mathrm{lo}}^{+\mathrm{hi}}$ in the table below.

### ULTRASAT, All-Sky Survey grid (AllSS)

In [ ]:
config = CONFIGS["ultrasat_allsky"]
prior = prior_for(config["events_file"])

latex_table = summarize_selected_detected_events(
    config["events_file"],
    quantiles=(0.5, 0.05, 0.95),
    merger_rate_lo=prior["lo"],
    merger_rate_mid=prior["mid"],
    merger_rate_hi=prior["hi"],
    run_duration=run_duration,
    aggregate_runs=O5_RUNS,
    aggregate_label="O5 total",
    poisson_lognormal_rate_quantiles=poisson_lognormal_rate_quantiles,
    output_file=config["table_file"],
    verbose=True,
)

### ULTRASAT, Low-Cadence Survey grid (LCS)

In [ ]:
config = CONFIGS["ultrasat_lcs"]
prior = prior_for(config["events_file"])

latex_table = summarize_selected_detected_events(
    config["events_file"],
    quantiles=(0.5, 0.05, 0.95),
    merger_rate_lo=prior["lo"],
    merger_rate_mid=prior["mid"],
    merger_rate_hi=prior["hi"],
    run_duration=run_duration,
    aggregate_runs=O5_RUNS,
    aggregate_label="O5 total",
    poisson_lognormal_rate_quantiles=poisson_lognormal_rate_quantiles,
    output_file=config["table_file"],
    verbose=True,
)

### UVEX

In [ ]:
config = CONFIGS["uvex"]
prior = prior_for(config["events_file"])

latex_table = summarize_selected_detected_events(
    config["events_file"],
    quantiles=(0.5, 0.05, 0.95),
    merger_rate_lo=prior["lo"],
    merger_rate_mid=prior["mid"],
    merger_rate_hi=prior["hi"],
    run_duration=run_duration,
    aggregate_runs=O5_RUNS,
    aggregate_label="O5 total",
    poisson_lognormal_rate_quantiles=poisson_lognormal_rate_quantiles,
    output_file=config["table_file"],
    verbose=True,
)

# Area-distance detection efficiency

Selected events in the (luminosity distance, localization area) plane: star
size encodes the scheduler objective and colour the detection probability,
with marginal histograms and the boundary of what the mission can reach
(maximum area, maximum distance, and the $d^{-4}$ segment between them).


## ULTRASAT, All-Sky Survey grid (AllSS)

In [ ]:
config = CONFIGS["ultrasat_allsky"]
plot_area_distance(config["events_file"], outdir=OUTPUTS, show=True)

## ULTRASAT, Low-Cadence Survey grid (LCS)

In [ ]:
config = CONFIGS["ultrasat_lcs"]
plot_area_distance(config["events_file"], outdir=OUTPUTS, show=True)

In [ ]:
def plot_objective_vs_detection(table, cutoff, run, figurename="obj-vs-det"):
    """
    Objective vs Detection with area as marker size.
    - X: objective_value
    - Y: detection_probability
    - Color: distance
    - Size: sky area (90% credible region)
    """

    triggered_table = table[table["objective_value"] >= cutoff]

    # ===================================================================
    # Extract data
    # ===================================================================
    obj_val = triggered_table["objective_value"]
    det_prob = triggered_table["detection_probability_known_position"]
    distance = triggered_table["distance"]
    area = triggered_table["area(90)"]

    # ===================================================================
    # Normalize area for marker size
    # ===================================================================
    # Small area => small marker, large area => large marker
    # Scale between 50 and 1000 for visibility
    size_min, size_max = 50, 2000
    area_normalized = (area - area.min()) / (area.max() - area.min())
    sizes = size_min + area_normalized * (size_max - size_min)

    # ===================================================================
    # Create figure
    # ===================================================================
    _, ax = plt.subplots(figsize=(10, 8))

    scatter = ax.scatter(
        obj_val,
        det_prob,
        s=sizes,  # <= Size proportional to area
        c=distance,
        cmap="viridis",
        alpha=0.7,  # Slightly more transparent for overlapping
        edgecolors="black",
        linewidth=0.8,
    )

    # Reference lines
    ax.axvline(
        cutoff,
        color="red",
        linestyle="--",
        linewidth=2.5,
        label=f"Cutoff = {cutoff:.2f}",
    )
    ax.axhline(0.5, color="orange", linestyle="--", linewidth=2, label="50% detection")
    ax.plot([0, 1], [0, 1], "k:", linewidth=1.5, alpha=0.3, label="Perfect correlation")

    # Labels
    ax.set_xlabel("Objective Value", fontsize=13, weight="bold")
    ax.set_ylabel("Detection Probability", fontsize=13, weight="bold")
    ax.set_title(
        f"{run}\nMarker size ∝ Sky localization area", fontsize=15, weight="bold"
    )
    ax.set_xlim(cutoff - 0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)

    # Legend
    ax.legend(loc="lower right", fontsize=10, framealpha=0.95)

    # ===================================================================
    # Colorbar for distance
    # ===================================================================
    plt.colorbar(scatter, ax=ax, label="Distance (Mpc)")

    # ===================================================================
    # Size legend for area
    # ===================================================================
    # Create manual legend for marker sizes
    from matplotlib.lines import Line2D

    # Representative sizes (small, medium, large)
    area_percentiles = np.percentile(area, [10, 50, 90])
    size_percentiles = np.percentile(sizes, [10, 50, 90])

    legend_elements = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="gray",
            markeredgecolor="black",
            markersize=np.sqrt(size_percentiles[0] / 4),
            label=rf"{area_percentiles[0]:.0f} deg$^2$",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="gray",
            markeredgecolor="black",
            markersize=np.sqrt(size_percentiles[1] / 4),
            label=rf"{area_percentiles[1]:.0f} deg$^2$",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="gray",
            markeredgecolor="black",
            markersize=np.sqrt(size_percentiles[2] / 4),
            label=rf"{area_percentiles[2]:.0f} deg$^2$",
        ),
    ]

    ax.legend(
        handles=legend_elements,
        loc="lower right",
        title="Sky Area (90%)",
        fontsize=9,
        framealpha=0.95,
        title_fontsize=10,
    )
    output_file = f"{figurename}_{run}.pdf"

    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    plt.show()

In [ ]:
config = CONFIGS["ultrasat_lcs"]
main_table = QTable.read(config["events_file"])

cutoff = main_table["cutoff"][0]
run = "O5c"
table = main_table[main_table["run"] == run]

plot_objective_vs_detection(table, cutoff, run, figurename=config["figure_prefix"])

## UVEX

In [ ]:
config = CONFIGS["uvex"]
plot_area_distance(config["events_file"], outdir=OUTPUTS, show=True)

In [ ]:
config = CONFIGS["uvex"]
main_table = QTable.read(config["events_file"])

cutoff = main_table["cutoff"][0]
run = "O5c"
table = main_table[main_table["run"] == run]

plot_objective_vs_detection(table, cutoff, run, figurename=config["figure_prefix"])

## Kilonova absolute magnitude, per mission

`absmag_mean` is the magnitude the source would have at 10 pc through the
observing filter. It depends on the spectrum and on the bandpass, not on the
telescope's sensitivity — that enters `limmag`. Comparing two missions
therefore means giving them the *same source*, evaluated in their *own* band.

We take the 8900 K blackbody fitted to AT2017gfo, anchor it on the $-16.0$
adopted for ULTRASAT, and integrate it through each `m4opt` bandpass. The
anchoring cancels the normalisation, so only the colour of the spectrum enters.

In [ ]:
import synphot
from astropy import units as u
from m4opt import missions

TEMPERATURE = 8900 * u.K
ANCHOR, ANCHOR_MAG = ("ultrasat", "NUV"), -16.0

spectrum = synphot.SourceSpectrum(synphot.BlackBodyNorm1D, temperature=TEMPERATURE)
raw = {
    (m, b): synphot.Observation(
        spectrum, getattr(missions, m).detector.bandpasses[b], force="extrap"
    )
    .effstim(u.ABmag)
    .value
    for m, b in [("ultrasat", "NUV"), ("uvex", "NUV"), ("uvex", "FUV")]
}
offset = ANCHOR_MAG - raw[ANCHOR]

for (m, b), v in raw.items():
    print(f"{m:9s} {b:4s}  M_AB = {v + offset:+.2f}")

ULTRASAT $-16.00$, UVEX NUV $-15.61$. The 0.39 mag between them comes from the
bandpass, not from the source. The UVEX value falls inside the
$[-17.8, -15.3]$ shock-powered NUV range of Kulkarni et al. (2021); the runs
use $-15.6$.

The FUV value, $-13.15$, is **not** usable: a single-temperature blackbody
fitted at NUV wavelengths cannot be extrapolated 1000 Å bluewards, and it lands
1.9 mag outside the corresponding $[-17.9, -15.0]$ range.

In [ ]:
config = CONFIGS["uvex"]

t = QTable.read(config["events_file"])
for k in ("snr", "delay", "absmag_mean", "absmag_stdev", "visits", "deadline"):
    print(k, {str(v) for v in t[k]})

## Numbers, table and figure of the paper

Every value quoted in Sections 4 and 5, Table 5 (`mission_comparison.tex`) and
Figure 4 (`detections_by_area_distance.pdf`) come from
`ultrasat_gw_followup.workflow.paper_results`, which is under test and can also
be run from the command line as `ugf-paper-results`. This cell only calls it.

The report lists each number by section of the paper and ends with a check of
its qualitative statements. A line marked `CHECK THE TEXT` means a new run
contradicts the prose.


In [ ]:
from IPython.display import Image, display

from ultrasat_gw_followup.workflow.paper_results import run as paper_results

numbers = paper_results(
    [
        "--lcs",
        str(CONFIGS["ultrasat_lcs"]["events_file"]),
        "--allsky",
        str(CONFIGS["ultrasat_allsky"]["events_file"]),
        "--uvex",
        str(CONFIGS["uvex"]["events_file"]),
        "--scenarios",
        str(RESULTS / "observing-scenarios.ecsv"),
        "--outdir",
        str(OUTPUTS),
    ]
)

In [ ]:
display(Image(filename=str(OUTPUTS / "detections_by_area_distance.png"), width=420))